# DGIdb Dataset Demo

This notebook demonstrates DGIdb exploration from the jupyter-node environment.

## Required environment variables
- `SIBLING_DGIDB_NODE_URL` (default used here: `http://dgidb-node:8005`)
- Optional: `JUPYTER_NODE_URL` (for display only)

## DGIdb endpoints used in this notebook
- `GET /health`
- `GET /ready`
- `GET /.well-known/a2a`
- `GET /data/summary`
- `GET /data/tables`
- `GET /dgidb/search?gene=...&drug=...&limit=...`
- `GET /mcp/tools`
- `POST /mcp/call`

## 0. Setup

In [ ]:
import os
import json
import socket
import importlib
from pprint import pprint

REQUIRED_PACKAGES = {
    "httpx": "httpx",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}

missing_packages = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    raise RuntimeError(
        "Missing packages detected: "
        f"{missing_packages}. Install them, then rerun this cell."
    )

import httpx
import pandas as pd
import matplotlib.pyplot as plt

JUPYTER_NODE_URL = os.environ.get("JUPYTER_NODE_URL", "http://localhost:8002")
DGIDB_NODE_URL = os.environ.get("SIBLING_DGIDB_NODE_URL", "http://dgidb-node:8005")

print(f"Jupyter node: {JUPYTER_NODE_URL}")
print(f"DGIdb node  : {DGIDB_NODE_URL}")

In [ ]:
# Preflight network and service checks.
dgidb_host = DGIDB_NODE_URL.replace("http://", "").replace("https://", "").split(":")[0]

try:
    socket.getaddrinfo(dgidb_host, None)
    print(f"DGIdb DNS: OK ({dgidb_host})")
except Exception as exc:
    print(f"DGIdb DNS: FAILED ({exc})")

for path in ["/health", "/ready", "/.well-known/a2a"]:
    try:
        resp = httpx.get(f"{DGIDB_NODE_URL}{path}", timeout=10)
        print(f"DGIdb {path}: {resp.status_code}")
    except Exception as exc:
        print(f"DGIdb {path}: FAILED ({exc})")

## 1. Read agent discovery card

In [ ]:
card_resp = httpx.get(f"{DGIDB_NODE_URL}/.well-known/a2a", timeout=10)
card_resp.raise_for_status()
card = card_resp.json()
print(json.dumps(card, indent=2))

## 2. Query DGIdb summary and schema

In [ ]:
summary_resp = httpx.get(f"{DGIDB_NODE_URL}/data/summary", timeout=10)
summary_resp.raise_for_status()
summary_data = summary_resp.json()
print("Summary payload:")
pprint(summary_data)

tables_resp = httpx.get(f"{DGIDB_NODE_URL}/data/tables", timeout=10)
tables_resp.raise_for_status()
tables_payload = tables_resp.json().get("tables", {})

table_rows = []
if isinstance(tables_payload, dict):
    summary_counts = summary_data.get("tables", {}) if isinstance(summary_data, dict) else {}
    for table_name, columns in tables_payload.items():
        table_rows.append(
            {
                "table": table_name,
                "row_count": summary_counts.get(table_name),
                "column_count": len(columns) if isinstance(columns, list) else None,
            }
        )
elif isinstance(tables_payload, list):
    for row in tables_payload:
        if isinstance(row, dict):
            table_rows.append(
                {
                    "table": row.get("table"),
                    "row_count": row.get("row_count"),
                    "column_count": len(row.get("columns", [])),
                }
            )

pd.DataFrame(table_rows).sort_values("table").reset_index(drop=True)

## 3. Example DGIdb search and MCP calls

In [ ]:
search_resp = httpx.get(
    f"{DGIDB_NODE_URL}/dgidb/search",
    params={"gene": "EGFR", "limit": 10},
    timeout=15,
)
search_resp.raise_for_status()
search_payload = search_resp.json()
search_rows = search_payload.get("rows", [])
print(f"Search results: {search_payload.get('count', 0)}")
display(pd.DataFrame(search_rows).head(10))

tools_resp = httpx.get(f"{DGIDB_NODE_URL}/mcp/tools", timeout=10)
tools_resp.raise_for_status()
print("Available MCP tools:")
display(pd.DataFrame(tools_resp.json().get("tools", []))[["name", "description"]])

mcp_call_resp = httpx.post(
    f"{DGIDB_NODE_URL}/mcp/call",
    json={"tool": "run_query", "args": {"sql": "SELECT gene_name, drug_name FROM interactions LIMIT 5"}},
    timeout=15,
)
mcp_call_resp.raise_for_status()
display(pd.DataFrame(mcp_call_resp.json().get("rows", [])))

In [ ]:
# Optional: plot table counts from /data/summary.
table_counts = summary_data.get("tables", {})
labels = list(table_counts.keys())
values = list(table_counts.values())

plt.figure(figsize=(6, 4))
plt.bar(labels, values)
plt.title("DGIdb Table Counts")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Troubleshooting

- If `/health` fails: confirm the DGIdb service is running (`make up` in `agentic-dataset-dgidb`).
- If DNS lookup fails for `dgidb-node`: ensure both services are on the same Docker network (`agentnet`).
- If `/ready` returns degraded: run migration + seed (`make fetch-seed`).
- If MCP calls fail: verify request payload shape: `{"tool": ..., "args": {...}}`.
- If data query errors with 400: only read-only `SELECT` SQL is allowed.
- To target a non-default endpoint, set `SIBLING_DGIDB_NODE_URL` before launching Jupyter.